Subject: ST 554 - Final Project

Name: Franklin Zhou

Date: 4/19/2026

# Fitting Your Model (50 pts)

**Create a Jupyter notebook for the modeling fitting part and the Streaming part below.**

- The file `power_ml_data.csv` is available at the URL: https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv
- You should read this data into a standard pandas data frame using the `pd.read_csv()` function.
- Convert this to a spark data frame
- We are going to treat the `Power_Zone_3` variable as our response variable.
- We can use all of the other variables as predictors. (Imagine we know that the `Power_Zone_3` reading is going to go offline in the future and we need to be able to predict that value appropriately.)

In [1]:
# Load packages and initiate spark session
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/19 14:48:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

In [ ]:
# Read data
ml_data = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
df = spark.createDataFrame(ml_data) # convert to spark sql data frame
df.show(5)

We want to fit an elastic net model using CV (no training/test split, just using CV on the data we’ve read in) with the steps below. 

The transformations below should each use an `MLlib` function that can be put into a pipeline

- The Hour column is likely not stored as a `DoubleType`. If it is not, use an SQL transformer to cast the variable as a `DoubleType`


In [3]:
# Load packages
from pyspark.ml.feature import SQLTransformer, VectorAssembler, Binarizer, OneHotEncoder, StringIndexer, PCA
from pyspark.ml import Pipeline


In [4]:
# Check the schema
df.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])

In [12]:
# Cast Hour to Double Type and rename Power_Zone_3 as label
cast_sql = SQLTransformer(
    statement = """
        SELECT *, CAST(Hour AS DOUBLE) AS Hour_Double
        FROM __THIS__
    """
)

- Binarize the `Hour` column based on the column being less than 6.5 or not (night vs day essentially)

In [8]:
# Binarize Hour_Double: 1 if Hour < 6.5 (night), 0 otherwise (day).
binarizer = Binarizer(
    inputCol = "Hour_Double",
    outputCol = "Hour_Bin",
    threshold = 6.5
)

- One-hot encode the Month column

In [13]:
# StringIndexer maps each month a numeric index
month_indexer = StringIndexer(
    inputCol = "Month",
    outputCol = "Month_idx"
)

# OneHotEncoder converts numeric index to binary vector
month_encoder = OneHotEncoder(
    inputCol = "Month_idx",
    outputCol = "Month_vector"
)

- Run a PCA fit on the `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows` columns.

     - To do this, I first used a `VectorAssembler()` call to place these variables in a column together for use with the `PCA()` estimator.
    
     - Once fitted, then you’ll have a PCA transformer we’ll use in our pipeline.
    
    - We’ll use two PCs in our transformation.

In [14]:
pca_assembler = VectorAssembler(
    inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], 
    outputCol = "PCA_input"
)

pca = PCA(
    k = 2, 
    inputCol = "PCA_input", 
    outputCol = "PCA_features"
)

- Rename your response variable as `label`

In [15]:
# Rename Power_Zone_3 to label
label_sql = SQLTransformer(
    statement = "SELECT *, Power_Zone_3 AS label FROM __THIS__"
)

- Use VectorAssembler() to put your predictors into a features. Use the
     - two fitted PCA features
     - binary `Hour` variable
     - `Power_Zone_1`
     - `Power_Zone_2`
     - `Month` indicator variables


In [16]:
# Combine all predictors into the 'features' vector column.
features_assembler = VectorAssembler(
    inputCols = ["pca_features", "Hour_bin", "Power_Zone_1", "Power_Zone_2", "Month_vector"],
    outputCol = "features"
)

- This ends the pipeline of transformations!

In [20]:
# Combine all transformation stages into a single Pipeline
#transformation_pipeline = Pipeline(stages=[cast_sql, binarizer, month_indexer, month_encoder, pca_assembler, pca, label_sql, features_assembler])

- Now you’ll then use the `CrossValidator()` function and the LinearRegression() function to fit an elastic net model.

    - You should do the following grid for the regParam and elasticNetParam: All combinations of
    
        - regParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
        
        - elasticNetParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
        
- Now fit the model using 5-fold CV with `rmse` as your criterion!
        

In [21]:
# Load packages
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

In [22]:
# Setup LinearRegression instance
lr = LinearRegression()

# Setup parameters grid
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# Setup pipeline
transformation_pipeline = Pipeline(stages=[cast_sql, binarizer, month_indexer, month_encoder, pca_assembler, pca, label_sql, features_assembler, lr])

# Create cross validation instance
crossval_lr = CrossValidator(estimator = transformation_pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName = 'rmse'),
                          numFolds = 5)

In [23]:
# Fit the cv model
cv_model = crossval_lr.fit(df)

26/04/19 16:07:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/19 16:07:20 WARN Instrumentation: [95b32077] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 16:07:23 WARN Instrumentation: [05d11cf0] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 16:07:24 WARN Instrumentation: [4dad499e] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 16:07:26 WARN Instrumentation: [38555dd3] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 16:07:27 WARN Instrumentation: [a904dce6] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 16:07:28 WARN Instrumentation: [1be4cc0b] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 16:07:30 WARN Instrumentation: [4a44fe13] regP

- Report the optimal values chosen for the tuning parameters

- Report the CV error

In [64]:
# Create a list contains RMSE value associate with parameters value
my_list = []
for i in range(len(paramGrid)):
    my_list.append([cv_model.avgMetrics[i], paramGrid[i].values()])

import numpy as np
# Convert to numpy array 
arrange = np.array(my_list)
# Sort by RMSE value
my_list_sorted = arrange[arrange[:, 0].argsort()]

# Print top 5 rows
print(my_list_sorted[:5])

[[2148.206687847316 dict_values([0.99, 0.05])]
 [2148.2067164575724 dict_values([1.0, 0.05])]
 [2148.2067496894183 dict_values([0.98, 0.05])]
 [2148.2067606811124 dict_values([0.95, 0.05])]
 [2148.206791086114 dict_values([0.9, 0.05])]]


From the output we find that the minumum RMSE is 2148.206687847316 while the best `regParam` value is 0.99 and the best `elasticNetParam` value is 0.05.

In [73]:
# Another way to extract the value
best_lr = cv_model.bestModel.stages[-1]
# Retrieve optimal tuning parameters
print("Best regParam value is:", best_lr._java_obj.getRegParam())
print("Best elasticNetParam value is:", best_lr._java_obj.getElasticNetParam())
# CV RMSE 
print("CV RMSE:", min(cv_model.avgMetrics))

Best regParam value is: 0.99
Best elasticNetParam value is: 0.05
CV RMSE: 2148.206687847316


- Report the training set RMSE (as done in the notes) by using your fitted model as a transformer and evaluating on the entire training set

In [77]:
# Now cv_model is a transformer with "best model" as default.
train_predictions = cv_model.transform(df)
training_rmse = RegressionEvaluator(metricName="rmse").evaluate(train_predictions)
print("The training RMSE value is:", training_rmse)

The training RMSE value is: 2147.0977796863613


- Take the outputted transformations from the model (the predictions) and create a `residual` column (`label` - `prediction`). The `.withColumn()` method is handy here. Print out a data frame with these `residual`s, the `label` column, and the `prediction`s

In [79]:
from pyspark.sql.functions import col
# create column residual = label - prediction
res_df = train_predictions.withColumn("residual", col("label") - col("prediction")) \
             .select("label","prediction","residual")

res_df.show(10)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20874.960071076963|-633.9962110769629|
|20131.08434| 18659.61962099528|1471.4647190047217|
|19668.43373| 18204.28819210263|1464.1455378973696|
|18899.27711| 17590.30198289228|1308.9751271077184|
|18442.40964|16997.138290385927| 1445.271349614075|
|18130.12048|16517.639478142202|1612.4810018577991|
|17945.06024|16093.292012174647|1851.7682278253524|
|17459.27711|15722.841661749717|1736.4354482502822|
|17025.54217|15271.297248679493|1754.2449213205073|
|16794.21687|14938.697443510253|1855.5194264897473|
+-----------+------------------+------------------+
only showing top 10 rows


# Reference

https://share.google/aimode/nADjDIlL2cBxKdAkP
